In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re
from datetime import datetime
from datetime import date
from dateutil.relativedelta import relativedelta


# Importa os dados

In [3]:
df = pd.read_excel(
    "dados_atualizado.xlsx",
    sheet_name=0,
    header=1,          # usa a 2ª linha como cabeçalho real
    dtype={"Class. Por idade": str},  # evita converter "1" em número se quiser
)


## Criando Pipeline de Limpeza dos Dados

In [5]:
def limpa_dados(df):
    
    colunas_data = ["data_nascimento","data_primeira_consulta","data_primeira_colonoscopia"]

    def padroniza_colunas(df):
        df.columns = (df.columns.astype(str).str.replace("\n", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip())
        return df

    def renomeia_colunas(df):
        df = df.rename(columns={
            "Nome": "nome",
            "Class. Por idade": "classificacao_idade",
            "Data nascimento": "data_nascimento",
            "Data 1ª consulta": "data_primeira_consulta",
            "Tempo entre início dos sintomas e primeira consulta": "intervalo_sintomas_primeira_consulta",
            "Sexo": "sexo",
            "Tipo": "tipo",
            "Perda de peso": "perda_peso",
            "Idade (meses) início dos sintomas": "intervalo_nascimento_sintomas",
            "Data 1ª colono": "data_primeira_colonoscopia",
            "Unnamed: 10": "intervalo_sintomas_primeira_colonoscopia",
            "Classificação": "classificacao",
        })
        return df

    def formata_colunas_data(df):
        for coluna in colunas_data:
            df[coluna] = (pd.to_datetime(df[coluna], errors="coerce").dt.strftime("%d/%m/%Y"))
        return df

    def deixar_coluna_inteira(df, coluna):
        for index, tempo in enumerate(df[coluna].tolist()):
            if tempo != '?':
                numero = int(re.sub(r"\D", "", tempo))
                df.loc[index, coluna] = numero
            elif tempo == "?":
                df.loc[index, coluna] = -1
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
            df[coluna] = df[coluna].astype("Int64")
        return df

        
    def criar_coluna(df, coluna):
        datas = pd.to_datetime(df['data_nascimento'],format="%d/%m/%Y",errors="coerce")
        meses = df["intervalo_nascimento_sintomas"]
        df[coluna] = [d + pd.DateOffset(months=int(m)) if pd.notna(d) and pd.notna(m) else pd.NaT for d, m in zip(datas, meses)  ]
        df[coluna] = (pd.to_datetime(df[coluna],errors="coerce" ).dt.strftime("%d/%m/%Y") )
        return df

    

    def ajustar_coluna_tipo(df):
        for index, tipo in enumerate(df['tipo'].tolist()):
            if tipo== 'Diarreia ':
                  df.loc[index, "tipo"] = 'Diarreia'
            
        return df
        
    def padronizar_sim_nao(df):
        for index, resposta in enumerate(df['perda_peso'].tolist()):
            if resposta == "não" or resposta == 'Não':
                  df.loc[index, "perda_peso"] = 'N'
            else:
                df.loc[index, "perda_peso"] = 'S'
        return df   


    def calcula_intervalo_colonoscopia(df):
        resultados = []
        for index in range(len(df)):
            d1_str = df.loc[index, "data_inicio_sintomas"]
            d2_str = df.loc[index, "data_primeira_colonoscopia"]
            # 1) converte para datetime
            try:
                d1 = datetime.strptime(d1_str, "%d/%m/%Y")
                d2 = datetime.strptime(d2_str, "%d/%m/%Y")
            except (ValueError, TypeError):
                resultados.append(None)
                continue
            # 2) valida ordem
            if d2 <= d1:
                resultados.append(None)
                continue
            # 3) calcula meses completos
            meses = ((d2.year - d1.year) * 12 + (d2.month - d1.month) - (d2.day < d1.day))

            resultados.append(meses)

        df["intervalo_sintomas_primeira_colonoscopia_recalculado"] = resultados
        df["intervalo_sintomas_primeira_colonoscopia_recalculado"] = pd.to_numeric(df["intervalo_sintomas_primeira_colonoscopia_recalculado"],errors="coerce")
        df["intervalo_sintomas_primeira_colonoscopia_recalculado"] = df["intervalo_sintomas_primeira_colonoscopia_recalculado"].astype("Int64")

        return df


    df = df.drop(index=0).reset_index(drop=True)
    df = padroniza_colunas(df)
    df = renomeia_colunas(df)
    df = formata_colunas_data(df)
    df = df.drop(columns=["nome"])
    df = deixar_coluna_inteira(df,"intervalo_sintomas_primeira_consulta")
    df = deixar_coluna_inteira(df,"intervalo_sintomas_primeira_colonoscopia")
    df = criar_coluna(df,"data_inicio_sintomas")  
    df = calcula_intervalo_colonoscopia(df)
    df = padronizar_sim_nao(df)
    df = ajustar_coluna_tipo(df)

    return df

In [6]:
dados_df = limpa_dados(df)
dados_df = dados_df[dados_df["intervalo_sintomas_primeira_consulta"]!=-1]

In [7]:
# dados_df['tipo'].unique()

In [8]:
## Fazer uma analise exploratoria montar alguns graficos por sexo

In [9]:
mulheres_df = dados_df[dados_df['sexo'] == 'F'].drop(columns=["sexo"])


homens_df = dados_df[dados_df['sexo'] == 'M'].drop(columns=["sexo"])

# Algumas metricas

In [11]:
colunas_nao_categoricas = [
    "data_nascimento",
    "data_primeira_consulta",
    "intervalo_sintomas_primeira_consulta", 
    "intervalo_nascimento_sintomas", 
    "data_primeira_colonoscopia",
    
    
    "intervalo_sintomas_primeira_colonoscopia",

    
    "data_inicio_sintomas",



    
    "intervalo_sintomas_primeira_colonoscopia_recalculado",
]

In [12]:
# classificacao_idade  = ['VEOIBD ', 'VEOIBD', 'DI Ped', 'Início precoce']

# tipo  = ['Enterorragia', 'Dor abdominal', 'Diarreia', 'Febre', 'Diarreia ',
       # 'Perda ponderal', 'Hematoquezia', 'Desaceleração do crescim.']


# perda_peso = ['não', 'Não', 'sim', 'Sim', 'SIm'] # Virou S ou N

# classificacao = ['CU', 'CNC', 'DC']

In [13]:
# resultado = df[(df['idade'] > 30) & (df['cidade'] == 'São Paulo')]

In [23]:
len(mulheres_colunas_categoricas_df[mulheres_colunas_categoricas_df['tipo']== 'Diarreia'])

NameError: name 'mulheres_colunas_categoricas_df' is not defined

In [ ]:
# A mesma coisa porem de jeitos diferentes

# mulheres_df[(mulheres_df['tipo']=='Diarreia') & (mulheres_df['perda_peso'] == 'S')]

pd.crosstab(
    mulheres_colunas_categoricas_df["tipo"],
    mulheres_colunas_categoricas_df["perda_peso"]
)

In [ ]:
cenarios = (
    mulheres_colunas_categoricas_df
    .groupby([
        "classificacao_idade",
        "tipo",
        "perda_peso",
        "classificacao"
    ])
    .size()
    .reset_index(name="quantidade")
    .sort_values("quantidade", ascending=False)
)

cenarios

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=mulheres_colunas_categoricas_df,
    x="tipo",
    hue="classificacao"
)

plt.xticks(rotation=45)
plt.xlabel("Tipo")
plt.ylabel("Quantidade")
plt.title("Classificação por tipo")
plt.tight_layout()

plt.show()

In [ ]:
ct = pd.crosstab(
    [
        mulheres_colunas_categoricas_df["classificacao_idade"],
        mulheres_colunas_categoricas_df["tipo"],
        mulheres_colunas_categoricas_df["perda_peso"]
    ],
    mulheres_colunas_categoricas_df["classificacao"]
)

ct

In [ ]:
cenario = ct.loc[
    ("DI Ped", slice(None), "S"),
    :
]

cenario.index = cenario.index.droplevel([0, 2])

plt.figure(figsize=(8, 5))

sns.heatmap(
    cenario,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("DI Ped — Perda de peso: S")
plt.xlabel("Classificação")
plt.ylabel("Tipo")

plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(
    [
        mulheres_colunas_categoricas_df["classificacao_idade"],
        mulheres_colunas_categoricas_df["tipo"],
        mulheres_colunas_categoricas_df["perda_peso"]
    ],
    mulheres_colunas_categoricas_df["classificacao"]
)

In [ ]:
mulheres_colunas_categoricas_df[mulheres_colunas_categoricas_df['perda_peso'] == 'S']

## Analise Mulheres

In [ ]:
# mulheres_df
media = mulheres_df["intervalo_nascimento_sintomas"].mean()
mulheres_colunas_categoricas_df = mulheres_df.drop(columns=colunas_nao_categoricas)

mulheres_colunas_categoricas_df['classificacao'].unique()

## Analise Homens

In [ ]:
# homens_df
media2 = homens_df["intervalo_nascimento_sintomas"].mean()
media2